<a href="https://colab.research.google.com/github/nibaskumar93n-debug/Morphoinformatics/blob/main/Key_bKGs_selection_SHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q",
                "xgboost", "shap", "scikit-learn",
                "pandas", "numpy", "matplotlib",
                "seaborn", "requests"], check=True)
print("✅ All packages installed!")


In [ ]:
from google.colab import files
import io

print("📁 Please upload:")
print("   → pred_metagenome_unstrat.tsv  (PICRUSt2 output)")
print("   → metadata.tsv                (Groups: DKD / HC)\n")

uploaded = files.upload()

file_map = {}
for fname in uploaded.keys():
    fl = fname.lower()
    if any(k in fl for k in ["pred", "metagenome", "unstrat", "ko", "picrust"]):
        file_map["picrust"] = fname
    elif any(k in fl for k in ["meta", "sample"]) or fl.endswith(".tsv"):
        file_map["metadata"] = fname

print("\n✅ Files detected:")
for k, v in file_map.items():
    print(f"   {k:10s} → {v}")
if len(file_map) < 2:
    print("\n⚠️  Auto-detection incomplete. Set names manually in Cell 3.")


In [ ]:
PICRUST_FILE   = file_map.get("picrust",  "pred_metagenome_unstrat.tsv")
METADATA_FILE  = file_map.get("metadata", "metadata.tsv")

TARGET_COLUMN  = "Groups"      # column name in metadata
CLASS_CASE     = "DKD"         # disease group label
CLASS_CTRL     = "HC"          # control group label

TOP_N          = 30            # genes shown in plots
SHAP_THRESHOLD = 0.100         # key gene cutoff
N_CV_FOLDS     = 5             # stratified k-fold
RANDOM_STATE   = 42
FETCH_KEGG     = True          # fetch gene names from KEGG REST API

# Permutation importance settings
N_PERMUTATIONS = 30            # repeats for permutation importance

print("✅ Configuration:")
print(f"   PICRUSt2 file  : {PICRUST_FILE}")
print(f"   Metadata file  : {METADATA_FILE}")
print(f"   Target         : '{TARGET_COLUMN}'  ({CLASS_CASE} vs {CLASS_CTRL})")
print(f"   SHAP threshold : ≥ {SHAP_THRESHOLD}")
print(f"   CV folds       : {N_CV_FOLDS}")
print(f"   Permutations   : {N_PERMUTATIONS}")


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, requests, time
warnings.filterwarnings("ignore")

from sklearn.model_selection    import StratifiedKFold
from sklearn.preprocessing      import LabelEncoder
from sklearn.metrics            import (roc_auc_score, roc_curve, auc,
                                        classification_report,
                                        ConfusionMatrixDisplay)
from sklearn.inspection         import permutation_importance
from sklearn.calibration        import calibration_curve, CalibratedClassifierCV
from xgboost                    import XGBClassifier
import shap

print("✅ All libraries imported!")

In [ ]:

def fetch_kegg_annotation(ko_ids, batch_size=10, pause=0.35):
    results  = {}
    ko_ids   = [k for k in ko_ids if str(k).startswith("K")]
    total    = len(ko_ids)
    print(f"Fetching KEGG annotations for {total} KO terms ...")

    for i in range(0, total, batch_size):
        batch = ko_ids[i : i + batch_size]
        try:
            resp = requests.get(
                f"https://rest.kegg.jp/get/{'+'.join(batch)}", timeout=20)
            if resp.status_code != 200:
                continue
            cur_ko, cur = None, {}
            for line in resp.text.splitlines():
                if line.startswith("ENTRY"):
                    if cur_ko:
                        results[cur_ko] = cur
                    cur_ko = line.split()[1].strip()
                    cur    = {"gene_name": "", "definition": "", "pathway": ""}
                elif line.startswith("NAME") and cur_ko:
                    cur["gene_name"] = line.replace("NAME","").strip().split(";")[0].strip()
                elif line.startswith("DEFINITION") and cur_ko:
                    cur["definition"] = line.replace("DEFINITION","").strip()
                elif cur_ko and line.startswith("  ") and cur.get("pathway","") == "":
                    parts = line.strip().split("  ")
                    if len(parts) >= 2 and parts[0].startswith("map"):
                        cur["pathway"] = parts[-1].strip()
            if cur_ko:
                results[cur_ko] = cur
        except Exception as e:
            print(f"   ⚠️  Batch {i//batch_size+1} error: {e}")
        time.sleep(pause)
        done = min(i + batch_size, total)
        if done % 50 == 0 or done == total:
            print(f"   {done}/{total} annotated ...")

    df = pd.DataFrame.from_dict(results, orient="index").reset_index()
    df.columns = ["KO", "gene_name", "definition", "pathway"]
    print(f"✅ Annotated {len(df)}/{total} KO terms")
    return df

print("✅ KEGG fetcher ready")

In [ ]:

gene_table = pd.read_csv(
    io.BytesIO(uploaded[PICRUST_FILE]),
    sep="\t", index_col=0, comment="#"
).T
gene_table = gene_table.apply(pd.to_numeric, errors="coerce").fillna(0)
print(f"Gene table (samples × genes): {gene_table.shape}")
print(f"  Sample IDs : {gene_table.index[:3].tolist()}")
print(f"  Gene IDs   : {gene_table.columns[:3].tolist()}")

# Metadata
metadata = pd.read_csv(
    io.BytesIO(uploaded[METADATA_FILE]),
    sep="\t", index_col=0
)
metadata.index = metadata.index.astype(str)
if str(metadata.index[0]).startswith("#"):      # skip QIIME2 type row
    metadata = metadata.iloc[1:]

print(f"\nMetadata      : {metadata.shape}")
print(f"  Columns     : {metadata.columns.tolist()}")
print(f"  Group counts: {metadata[TARGET_COLUMN].value_counts().to_dict()}")


In [ ]:
gene_table.index = gene_table.index.astype(str)
metadata.index   = metadata.index.astype(str)

common = gene_table.index.intersection(metadata.index)
print(f"Common samples: {len(common)}")
if len(common) == 0:
    raise ValueError("❌ No common samples found! Check sample ID formatting.")

gene_table = gene_table.loc[common]
metadata   = metadata.loc[common].dropna(subset=[TARGET_COLUMN])
gene_table = gene_table.loc[metadata.index]
print(f"Samples after dropping missing target: {len(gene_table)}")

# Relative abundance → log(CPM + 1)
gene_table = gene_table.div(gene_table.sum(axis=1), axis=0)
gene_table = np.log1p(gene_table * 1e6)

# Drop zero-variance genes
gene_table = gene_table.loc[:, gene_table.var() > 0]
print(f"Genes after zero-variance filter: {gene_table.shape[1]}")

# Encode labels
X           = gene_table.values.astype(np.float32)
gene_names  = gene_table.columns.tolist()
le          = LabelEncoder()
y           = le.fit_transform(metadata[TARGET_COLUMN].astype(str).values)
classes     = le.classes_
n_classes   = len(classes)

print(f"\nFeature matrix : {X.shape}")
print(f"Classes        : {dict(zip(range(n_classes), classes))}")
print(f"Class counts   : {dict(zip(classes, np.bincount(y)))}")


In [ ]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 8 — Robust CV loop
#   FIX 1: SHAP on held-out test fold only  (no leakage)
#   FIX 2: Feature importance stability across folds
#   FIX 4: ROC curve per fold
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title 🤖 Cell 8: Robust cross-validation loop (SHAP leakage-free)
# In[8]:
xgb_params = dict(
    n_estimators     = 500,
    max_depth        = 4,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    eval_metric      = "mlogloss" if n_classes > 2 else "logloss",
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
)

cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Storage across folds
fold_aucs         = []
fold_shap         = np.zeros((len(X), len(gene_names)))  # accumulate SHAP per sample
fold_gain_ranks   = []   # gain importance rank per fold  (FIX 2)
fold_shap_ranks   = []   # SHAP rank per fold             (FIX 2)
fold_roc          = []   # (fpr, tpr, auc) per fold       (FIX 4)
oof_proba         = np.zeros((len(X), n_classes))        # out-of-fold probabilities
fold_sample_count = np.zeros(len(X), dtype=int)          # how many times each sample was in test

print(f"Running {N_CV_FOLDS}-fold stratified CV ...\n")

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    # ── Train on train fold only ───────────────────────────────────
    fold_model = XGBClassifier(**xgb_params)
    fold_model.fit(X_tr, y_tr,
                   eval_set        = [(X_te, y_te)],
                   early_stopping_rounds = 30,
                   verbose         = False)

    # ── Predict on held-out test fold ─────────────────────────────
    proba_te   = fold_model.predict_proba(X_te)
    oof_proba[test_idx] = proba_te
    fold_sample_count[test_idx] += 1

    # AUC
    if n_classes == 2:
        fold_auc = roc_auc_score(y_te, proba_te[:, 1])
        fpr, tpr, _ = roc_curve(y_te, proba_te[:, 1])
    else:
        fold_auc = roc_auc_score(y_te, proba_te, multi_class="ovr")
        fpr, tpr = np.array([0, 1]), np.array([0, 1])  # placeholder for multi-class
    fold_aucs.append(fold_auc)
    fold_roc.append((fpr, tpr, fold_auc))

    # ── FIX 1: SHAP on test fold only ─────────────────────────────
    explainer_fold  = shap.TreeExplainer(fold_model)
    shap_vals_fold  = explainer_fold.shap_values(X_te)

    if isinstance(shap_vals_fold, list):
        # multi-class: average absolute across classes
        abs_shap_te = np.mean([np.abs(sv) for sv in shap_vals_fold], axis=0)
    else:
        abs_shap_te = np.abs(shap_vals_fold)

    fold_shap[test_idx] += abs_shap_te   # accumulate

    # ── FIX 2: Gain rank this fold ────────────────────────────────
    gain_imp = fold_model.feature_importances_
    gain_rank = pd.Series(gain_imp, index=gene_names).rank(ascending=False)
    fold_gain_ranks.append(gain_rank)

    shap_mean_fold = abs_shap_te.mean(axis=0)
    shap_rank = pd.Series(shap_mean_fold, index=gene_names).rank(ascending=False)
    fold_shap_ranks.append(shap_rank)

    print(f"  Fold {fold}/{N_CV_FOLDS}  |  AUC: {fold_auc:.4f}  "
          f"|  best iteration: {fold_model.best_iteration}")

# ── Aggregate OOF SHAP (each sample's SHAP from its test fold) ───
# Normalise by fold count (always 1 for standard CV, but safe)
oof_shap_matrix = fold_shap / np.maximum(fold_sample_count[:, None], 1)
mean_oof_shap   = oof_shap_matrix.mean(axis=0)   # mean |SHAP| per gene

# ── Aggregate stability ranks (FIX 2) ────────────────────────────
gain_rank_df     = pd.DataFrame(fold_gain_ranks)   # shape: folds × genes
shap_rank_df     = pd.DataFrame(fold_shap_ranks)

mean_gain_rank   = gain_rank_df.mean()
std_gain_rank    = gain_rank_df.std()
mean_shap_rank   = shap_rank_df.mean()
std_shap_rank    = shap_rank_df.std()

print(f"\n{'─'*50}")
print(f"OOF AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"Per-fold: {[round(a,4) for a in fold_aucs]}")



In [ ]:
# CELL 9 — Build SHAP results dataframe + stability metrics
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title 📋 Cell 9: Build SHAP + stability dataframe
# In[9]:

# Directional SHAP: fit final model on full data for direction only
# (direction is intrinsically stable; magnitude already computed OOF)
final_model = XGBClassifier(**xgb_params)
final_model.fit(X, y)
final_explainer = shap.TreeExplainer(final_model)
final_shap      = final_explainer.shap_values(X)

if isinstance(final_shap, list):
    # class index 1 = DKD (if DKD is encoded as 1 by LabelEncoder)
    dkd_idx     = list(classes).index(CLASS_CASE) if CLASS_CASE in classes else 1
    dir_shap    = final_shap[dkd_idx].mean(axis=0)
else:
    dir_shap    = final_shap.mean(axis=0)

# Master dataframe
shap_df = pd.DataFrame({
    "KO"               : gene_names,
    "Mean_OOF_SHAP"    : mean_oof_shap,       # leakage-free magnitude
    "Directional_SHAP" : dir_shap,             # sign = direction in DKD
    "Mean_SHAP_Rank"   : mean_shap_rank.values,
    "Std_SHAP_Rank"    : std_shap_rank.values,
    "Mean_Gain_Rank"   : mean_gain_rank.values,
    "Std_Gain_Rank"    : std_gain_rank.values,
}).sort_values("Mean_OOF_SHAP", ascending=False).reset_index(drop=True)

# Stability score: lower std rank = more stable (invert so higher = better)
shap_df["Stability_Score"] = 1 / (1 + shap_df["Std_SHAP_Rank"])

shap_df["Direction_DKD"] = shap_df["Directional_SHAP"].apply(
    lambda v: f"↑ Enriched in {CLASS_CASE}" if v > 0 else f"↓ Depleted in {CLASS_CASE}"
)

print("✅ SHAP dataframe built (leakage-free OOF SHAP values)")
print(f"\nTop 10 genes:")
print(shap_df[["KO","Mean_OOF_SHAP","Stability_Score","Direction_DKD",
               "Mean_SHAP_Rank","Std_SHAP_Rank"]].head(10).to_string(index=False))

In [ ]:
# CELL 10 — FIX 3: Permutation importance
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title 🔀 Cell 10: Permutation importance (unbiased cross-check)
# In[10]:
print(f"Computing permutation importance ({N_PERMUTATIONS} repeats) ...")
print("This may take a few minutes ...\n")

perm_result = permutation_importance(
    final_model, X, y,
    n_repeats    = N_PERMUTATIONS,
    scoring      = "roc_auc_ovr" if n_classes > 2 else "roc_auc",
    random_state = RANDOM_STATE,
    n_jobs       = -1,
)

perm_df = pd.DataFrame({
    "KO"              : gene_names,
    "Perm_Importance" : perm_result.importances_mean,
    "Perm_Std"        : perm_result.importances_std,
}).sort_values("Perm_Importance", ascending=False).reset_index(drop=True)

# Merge permutation into main shap_df
shap_df = shap_df.merge(
    perm_df[["KO","Perm_Importance","Perm_Std"]],
    on="KO", how="left"
)

# Consensus rank: average of SHAP rank and Permutation rank
perm_rank = perm_df.set_index("KO")["Perm_Importance"].rank(ascending=False)
shap_df["Perm_Rank"]      = shap_df["KO"].map(perm_rank)
shap_df["Consensus_Rank"] = (shap_df["Mean_SHAP_Rank"] + shap_df["Perm_Rank"]) / 2

print("✅ Permutation importance computed")
print(f"\nTop 10 genes by Permutation Importance:")
print(perm_df[["KO","Perm_Importance","Perm_Std"]].head(10).to_string(index=False))



In [ ]:
# CELL 11 — KEGG annotation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title 🧬 Cell 11: KEGG annotation mapping
# In[11]:
if FETCH_KEGG:
    top_ko_ids = shap_df.head(max(TOP_N * 2, 100))["KO"].tolist()
    kegg_annot = fetch_kegg_annotation(top_ko_ids)
    shap_df    = shap_df.merge(kegg_annot, on="KO", how="left")
    shap_df["gene_name"]  = shap_df["gene_name"].fillna("Unknown")
    shap_df["definition"] = shap_df["definition"].fillna("No annotation")
    shap_df["pathway"]    = shap_df["pathway"].fillna("Unclassified")
else:
    shap_df["gene_name"]  = "Unknown"
    shap_df["definition"] = "No annotation"
    shap_df["pathway"]    = "Unclassified"

shap_df["Label"] = shap_df.apply(
    lambda r: r["gene_name"] if r["gene_name"] not in ["", "Unknown"]
              else r["KO"], axis=1
)

# ── Key gene tiering ──────────────────────────────────────────────
shap_df["Tier"] = pd.cut(
    shap_df["Mean_OOF_SHAP"],
    bins   = [-np.inf, 0.010, 0.050, 0.100, np.inf],
    labels = ["Negligible (<0.010)", "Low (0.010–0.049)",
              "Moderate (0.050–0.099)", "Key gene (≥0.100)"]
)

key_genes = shap_df[shap_df["Mean_OOF_SHAP"] >= SHAP_THRESHOLD].copy()

print(f"\nTier distribution:")
print(shap_df["Tier"].value_counts().sort_index().to_string())
print(f"\n🔑 KEY GENES (OOF SHAP ≥ {SHAP_THRESHOLD}): {len(key_genes)}")
if len(key_genes):
    print(key_genes[["KO","gene_name","Mean_OOF_SHAP",
                      "Stability_Score","Direction_DKD","pathway"]].to_string(index=False))


In [ ]:
# CELL 12 — FIX 4: ROC curve (per-fold + mean)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title 📈 Cell 12: ROC curves — per fold + mean ± SD
# In[12]:
if n_classes == 2:
    fig, ax = plt.subplots(figsize=(8, 7))
    mean_fpr    = np.linspace(0, 1, 300)
    interp_tprs = []

    colors = plt.cm.tab10(np.linspace(0, 0.5, N_CV_FOLDS))
    for fold_i, (fpr, tpr, fold_auc) in enumerate(fold_roc):
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        interp_tprs.append(interp_tpr)
        ax.plot(fpr, tpr, alpha=0.4, lw=1.5, color=colors[fold_i],
                label=f"Fold {fold_i+1}  (AUC = {fold_auc:.3f})")

    # Mean ROC
    mean_tpr    = np.mean(interp_tprs, axis=0)
    mean_tpr[-1]= 1.0
    std_tpr     = np.std(interp_tprs, axis=0)
    mean_auc    = auc(mean_fpr, mean_tpr)

    ax.plot(mean_fpr, mean_tpr, color="black", lw=2.5,
            label=f"Mean ROC (AUC = {mean_auc:.3f} ± {np.std(fold_aucs):.3f})")

    # Confidence band ± 1 SD
    tpr_upper = np.minimum(mean_tpr + std_tpr, 1)
    tpr_lower = np.maximum(mean_tpr - std_tpr, 0)
    ax.fill_between(mean_fpr, tpr_lower, tpr_upper,
                    color="grey", alpha=0.20, label="± 1 SD")

    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6, label="Random classifier")
    ax.set_xlabel("False Positive Rate", fontsize=13)
    ax.set_ylabel("True Positive Rate", fontsize=13)
    ax.set_title(f"ROC Curves — {N_CV_FOLDS}-Fold CV\n"
                 f"DKD vs HC  |  Mean AUC = {mean_auc:.3f}",
                 fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", fontsize=9)
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.05])
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(alpha=0.3, linestyle="--")

    plt.tight_layout()
    plt.savefig("roc_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Saved: roc_curves.png")
else:
    print("⚠️  Per-class ROC for multi-class not shown here (OVR AUC already computed).")



In [ ]:
# CELL 13 — FIX 5: Calibration curve
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title 📐 Cell 13: Calibration curve (OOF probabilities)
# In[13]:
if n_classes == 2:
    oof_proba_pos = oof_proba[:, 1]   # probability of DKD class

    # Raw model calibration
    prob_true_raw, prob_pred_raw = calibration_curve(
        y, oof_proba_pos, n_bins=10, strategy="quantile"
    )

    # Isotonic-calibrated model (fit on full data for reference)
    cal_model = CalibratedClassifierCV(
        XGBClassifier(**xgb_params), method="isotonic", cv=5
    )
    cal_model.fit(X, y)
    cal_proba = cal_model.predict_proba(X)[:, 1]
    prob_true_cal, prob_pred_cal = calibration_curve(
        y, cal_proba, n_bins=10, strategy="quantile"
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ── Reliability diagram ────────────────────────────────────────
    axes[0].plot([0,1], [0,1], "k--", lw=1.5, label="Perfect calibration")
    axes[0].plot(prob_pred_raw, prob_true_raw, "o-",
                 color="#E84040", lw=2, ms=7, label="XGBoost (raw OOF)")
    axes[0].plot(prob_pred_cal, prob_true_cal, "s-",
                 color="#4A90D9", lw=2, ms=7, label="XGBoost (isotonic calibrated)")
    axes[0].set_xlabel("Mean Predicted Probability", fontsize=12)
    axes[0].set_ylabel("Fraction of Positives (DKD)", fontsize=12)
    axes[0].set_title("Calibration Curve\n(Reliability Diagram)",
                      fontsize=13, fontweight="bold")
    axes[0].legend(fontsize=9)
    axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1])
    axes[0].grid(alpha=0.3, linestyle="--")
    axes[0].spines[["top","right"]].set_visible(False)

    # ── Histogram of predicted probabilities ──────────────────────
    dkd_mask = y == list(classes).index(CLASS_CASE)
    axes[1].hist(oof_proba_pos[dkd_mask],  bins=25, alpha=0.7,
                 color="#E84040", label=f"{CLASS_CASE}", density=True)
    axes[1].hist(oof_proba_pos[~dkd_mask], bins=25, alpha=0.7,
                 color="#4A90D9", label=f"{CLASS_CTRL}", density=True)
    axes[1].axvline(0.5, color="black", linestyle="--", lw=1.5, label="Decision boundary")
    axes[1].set_xlabel("Predicted Probability of DKD", fontsize=12)
    axes[1].set_ylabel("Density", fontsize=12)
    axes[1].set_title("OOF Predicted Probability Distribution\n(DKD vs HC)",
                      fontsize=13, fontweight="bold")
    axes[1].legend(fontsize=9)
    axes[1].spines[["top","right"]].set_visible(False)
    axes[1].grid(alpha=0.3, linestyle="--")

    plt.tight_layout()
    plt.savefig("calibration_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Saved: calibration_curve.png")
else:
    print("⚠️  Calibration curve shown for binary classification only.")



In [ ]:
# CELL 14 — Plot: SHAP importance bar (coloured by direction)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title 📊 Cell 14: SHAP bar chart — direction coloured
# In[14]:
top_plot = shap_df.head(TOP_N).sort_values("Mean_OOF_SHAP").copy()
bar_colors = ["#E84040" if v > 0 else "#4A90D9"
               for v in top_plot["Directional_SHAP"]]

fig, ax = plt.subplots(figsize=(13, 11))
ax.barh(top_plot["Label"], top_plot["Mean_OOF_SHAP"],
        color=bar_colors, edgecolor="white", linewidth=0.6)
ax.axvline(SHAP_THRESHOLD, color="black", linestyle="--", lw=1.8, alpha=0.8,
           label=f"Key gene threshold ({SHAP_THRESHOLD})")

for i, (_, row) in enumerate(top_plot.iterrows()):
    ax.text(row["Mean_OOF_SHAP"] + 0.001, i,
            f"  {row['KO']}  [stability={row['Stability_Score']:.2f}]",
            va="center", fontsize=6.5, color="dimgrey")

patch_up   = mpatches.Patch(color="#E84040", label=f"↑ Enriched in {CLASS_CASE}")
patch_down = mpatches.Patch(color="#4A90D9", label=f"↓ Depleted in {CLASS_CASE}")
thresh_line = plt.Line2D([0],[0], color="black", linestyle="--",
                          lw=1.8, label=f"Threshold ({SHAP_THRESHOLD})")
ax.legend(handles=[patch_up, patch_down, thresh_line],
          loc="lower right", fontsize=9)

ax.set_xlabel("Mean |OOF SHAP Value|  (leakage-free)", fontsize=13)
ax.set_title(f"Top {TOP_N} Key Genes — XGBoost + SHAP  (DKD vs HC)\n"
             f"OOF AUC: {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}  "
             f"|  Samples: {len(X)}",
             fontsize=14, fontweight="bold", pad=12)
ax.tick_params(axis="y", labelsize=9)
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("keygenes_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: keygenes_shap_bar.png")




In [ ]:
# CELL 15 — Plot: Stability heatmap across folds
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title 🔁 Cell 15: Feature stability heatmap across folds
# In[15]:
top30_genes = shap_df.head(TOP_N)["KO"].tolist()
top30_lbls  = shap_df.head(TOP_N)["Label"].tolist()

# Rank matrix: folds × top genes
rank_matrix = shap_rank_df[top30_genes].values.T  # genes × folds

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    rank_matrix,
    xticklabels = [f"Fold {i+1}" for i in range(N_CV_FOLDS)],
    yticklabels = top30_lbls,
    cmap        = "YlOrRd_r",          # low rank (=important) = dark
    annot       = True, fmt=".0f",
    annot_kws   = {"size": 8},
    linewidths  = 0.5,
    ax          = ax,
    cbar_kws    = {"label": "SHAP Rank (1 = most important)"}
)
ax.set_title(f"Feature Importance Stability — SHAP Rank per Fold\n"
             f"Top {TOP_N} Genes  |  Dark = consistently important",
             fontsize=13, fontweight="bold", pad=12)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.savefig("stability_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: stability_heatmap.png")


In [ ]:
# CELL 16 — Plot: SHAP vs Permutation importance (consensus)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title ⚖️ Cell 16: SHAP vs Permutation importance scatter
# In[16]:
top_scatter = shap_df.head(TOP_N).copy()

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(
    top_scatter["Mean_OOF_SHAP"],
    top_scatter["Perm_Importance"],
    c     = top_scatter["Consensus_Rank"],
    cmap  = "viridis_r",
    s     = 120, edgecolors="white", linewidths=0.8, zorder=3
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Consensus Rank (lower = better)", fontsize=10)

# Label top 15 points
for _, row in top_scatter.head(15).iterrows():
    ax.annotate(row["Label"],
                xy=(row["Mean_OOF_SHAP"], row["Perm_Importance"]),
                xytext=(5, 4), textcoords="offset points",
                fontsize=7, color="dimgrey")

ax.axvline(SHAP_THRESHOLD, color="#E84040", linestyle="--",
           lw=1.5, alpha=0.7, label=f"SHAP threshold ({SHAP_THRESHOLD})")
ax.axhline(0, color="grey", linestyle="--", lw=1, alpha=0.5)
ax.set_xlabel("Mean |OOF SHAP Value|  (leakage-free)", fontsize=12)
ax.set_ylabel("Permutation Importance (AUC drop)", fontsize=12)
ax.set_title(f"SHAP vs Permutation Importance\n"
             f"Top {TOP_N} Genes — Consensus view (DKD vs HC)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, linestyle="--")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("shap_vs_permutation.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: shap_vs_permutation.png")


In [ ]:
# CELL 17 — Plot: SHAP beeswarm
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title 🐝 Cell 17: SHAP beeswarm
# In[17]:
top_fids = shap_df.head(TOP_N)["KO"].tolist()
top_idx  = [gene_names.index(g) for g in top_fids if g in gene_names]
top_lbls = [shap_df.loc[shap_df["KO"] == gene_names[i], "Label"].values[0]
             for i in top_idx]

if isinstance(final_shap, list):
    dkd_idx        = list(classes).index(CLASS_CASE) if CLASS_CASE in classes else 1
    shap_plot_vals = final_shap[dkd_idx][:, top_idx]
    base_val       = (explainer.expected_value[dkd_idx]
                      if hasattr(final_explainer.expected_value, "__len__")
                      else final_explainer.expected_value)
else:
    shap_plot_vals = final_shap[:, top_idx]
    base_val       = final_explainer.expected_value

explanation = shap.Explanation(
    values        = shap_plot_vals,
    base_values   = base_val,
    data          = X[:, top_idx],
    feature_names = top_lbls
)

plt.figure(figsize=(12, 11))
shap.plots.beeswarm(explanation, show=False, max_display=TOP_N)
plt.title(f"SHAP Beeswarm — Top {TOP_N} Key Genes\n"
          f"Red = high abundance  |  Blue = low abundance",
          fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("keygenes_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: keygenes_shap_beeswarm.png")


In [ ]:
# CELL 18 — Plot: Pathway summary + Confusion matrix
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title 🗺️ Cell 18: Pathway summary + Confusion matrix
# In[18]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Pathway bar ───────────────────────────────────────────────────
pathway_df = (
    shap_df[shap_df["pathway"] != "Unclassified"]
    .groupby("pathway")["Mean_OOF_SHAP"]
    .sum().sort_values(ascending=False).head(15).reset_index()
)
if len(pathway_df):
    axes[0].bar(range(len(pathway_df)), pathway_df["Mean_OOF_SHAP"],
                color=sns.color_palette("tab20", len(pathway_df)),
                edgecolor="white", linewidth=0.7)
    axes[0].set_xticks(range(len(pathway_df)))
    axes[0].set_xticklabels(pathway_df["pathway"],
                             rotation=40, ha="right", fontsize=8)
    axes[0].set_ylabel("Cumulative Mean |OOF SHAP|", fontsize=11)
    axes[0].set_title("Top Metabolic Pathways\n(Cumulative OOF SHAP)",
                       fontsize=12, fontweight="bold")
    axes[0].grid(axis="y", alpha=0.3, linestyle="--")
    axes[0].spines[["top","right"]].set_visible(False)
else:
    axes[0].text(0.5, 0.5, "No pathway data\n(KEGG fetch off)",
                 ha="center", va="center", transform=axes[0].transAxes)

# ── Confusion matrix ──────────────────────────────────────────────
y_pred = final_model.predict(X)
ConfusionMatrixDisplay.from_predictions(
    y, y_pred, display_labels=classes,
    cmap="Blues", ax=axes[1], colorbar=False
)
axes[1].set_title("Confusion Matrix (final model, training data)",
                   fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("pathway_and_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nClassification Report (final model):")
print(classification_report(y, y_pred, target_names=classes))
print("✅ Saved: pathway_and_confusion.png")


In [ ]:
# CELL 19 — Save CSVs & download everything
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# @title 💾 Cell 19: Save CSVs & download all results
# In[19]:
from google.colab import files as colab_files

# ── Save CSVs ─────────────────────────────────────────────────────
shap_df.to_csv("all_genes_results.csv", index=False)
key_genes.to_csv("key_genes_above_threshold.csv", index=False)
perm_df.head(TOP_N).to_csv("permutation_importance_top.csv", index=False)
print("✅ CSVs saved")

# ── Final summary ─────────────────────────────────────────────────
print("\n" + "═"*62)
print("  KEY GENE DISCOVERY — FINAL SUMMARY")
print("═"*62)
print(f"  Samples                : {len(X)}")
print(f"  Total KO genes tested  : {len(gene_names)}")
print(f"  Target                 : {TARGET_COLUMN}  ({CLASS_CASE} vs {CLASS_CTRL})")
print(f"  OOF AUC                : {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
print(f"  SHAP leakage-free      : ✅  (computed on held-out test folds)")
print(f"  Stability checked      : ✅  (rank SD across {N_CV_FOLDS} folds)")
print(f"  Permutation confirmed  : ✅  ({N_PERMUTATIONS} repeats)")
print(f"  Key genes (≥{SHAP_THRESHOLD})  : {len(key_genes)}")
print(f"\n  🔑 KEY GENES (sorted by OOF SHAP):")
print("  " + "─"*58)
disp = ["KO","gene_name","Mean_OOF_SHAP","Stability_Score",
        "Direction_DKD","Perm_Importance","pathway"]
print(key_genes[disp].to_string(index=False))
print("═"*62)

# ── Download ──────────────────────────────────────────────────────
print("\n📥 Downloading all result files ...")
for fname in [
    "all_genes_results.csv",
    "key_genes_above_threshold.csv",
    "permutation_importance_top.csv",
    "roc_curves.png",
    "calibration_curve.png",
    "keygenes_shap_bar.png",
    "stability_heatmap.png",
    "shap_vs_permutation.png",
    "keygenes_shap_beeswarm.png",
    "pathway_and_confusion.png",
]:
    try:
        colab_files.download(fname)
        print(f"   ✅ {fname}")
    except Exception as e:
        print(f"   ⚠️  {fname} — {e}")